In [1]:
import pandas as pd
import numpy as np

print("Loading dataset...")

df = pd.read_csv(
    "labeled_transactions_april_sept.csv"
)

print(df.shape)

df = df.sort_values(
    "timestamp"
).reset_index(drop=True)

print("Sorted by timestamp.")

Loading dataset...
(2713386, 9)
Sorted by timestamp.


In [2]:
from collections import defaultdict

print("="*60)
print("TEMPORAL FEATURE ENGINEERING")
print("="*60)

n = len(df)

# ============================================================
# OUTPUT ARRAYS
# ============================================================

src_tx_count_past = np.zeros(n, dtype=np.int32)
dst_tx_count_past = np.zeros(n, dtype=np.int32)

src_to_dst_count_past = np.zeros(n, dtype=np.int32)
dst_to_src_count_past = np.zeros(n, dtype=np.int32)

time_since_src_last = np.zeros(n, dtype=np.float32)
time_since_dst_last = np.zeros(n, dtype=np.float32)

src_value_sum_past = np.zeros(n, dtype=np.float32)
dst_value_sum_past = np.zeros(n, dtype=np.float32)

# ============================================================
# HISTORICAL STATE
# ============================================================

wallet_tx_count = defaultdict(int)
wallet_value_sum = defaultdict(float)
wallet_last_time = {}

pair_count = defaultdict(int)

# ============================================================
# STREAM THROUGH TRANSACTIONS
# ============================================================

for i, row in enumerate(df.itertuples(index=False)):

    src = row.from_address
    dst = row.to_address
    ts = row.timestamp
    val = row.transaction_value

    # --------------------------------------------------------
    # wallet history
    # --------------------------------------------------------

    src_tx_count_past[i] = wallet_tx_count[src]
    dst_tx_count_past[i] = wallet_tx_count[dst]

    src_value_sum_past[i] = wallet_value_sum[src]
    dst_value_sum_past[i] = wallet_value_sum[dst]

    # --------------------------------------------------------
    # pair history
    # --------------------------------------------------------

    src_to_dst_count_past[i] = pair_count[(src, dst)]
    dst_to_src_count_past[i] = pair_count[(dst, src)]

    # --------------------------------------------------------
    # recency
    # --------------------------------------------------------

    if src in wallet_last_time:
        time_since_src_last[i] = (
            ts - wallet_last_time[src]
        )

    if dst in wallet_last_time:
        time_since_dst_last[i] = (
            ts - wallet_last_time[dst]
        )

    # --------------------------------------------------------
    # UPDATE HISTORY
    # --------------------------------------------------------

    wallet_tx_count[src] += 1
    wallet_tx_count[dst] += 1

    wallet_value_sum[src] += val
    wallet_value_sum[dst] += val

    wallet_last_time[src] = ts
    wallet_last_time[dst] = ts

    pair_count[(src, dst)] += 1

    # progress

    if i % 500000 == 0 and i > 0:
        print(f"{i:,}/{n:,}")

print("Done.")

TEMPORAL FEATURE ENGINEERING
500,000/2,713,386
1,000,000/2,713,386
1,500,000/2,713,386
2,000,000/2,713,386
2,500,000/2,713,386
Done.


In [8]:
df["src_tx_count_past"] = src_tx_count_past
df["dst_tx_count_past"] = dst_tx_count_past

df["src_to_dst_count_past"] = src_to_dst_count_past
df["dst_to_src_count_past"] = dst_to_src_count_past

df["time_since_src_last"] = time_since_src_last
df["time_since_dst_last"] = time_since_dst_last

df["src_value_sum_past"] = src_value_sum_past
df["dst_value_sum_past"] = dst_value_sum_past

df["log_transaction_value"] = np.log1p(
    df["transaction_value"]
)

print(df.shape)

(2713386, 18)


In [9]:
# ============================================================
# LOG TRANSFORM
# ============================================================

log_features = [
    "src_tx_count_past",
    "dst_tx_count_past",
    "src_to_dst_count_past",
    "dst_to_src_count_past",
    "time_since_src_last",
    "time_since_dst_last",
    "src_value_sum_past",
    "dst_value_sum_past"
]

for col in log_features:
    df[f"log_{col}"] = np.log1p(df[col])

df["log_transaction_value"] = np.log1p(
    df["transaction_value"]
)

print("Log features created.")

Log features created.


In [10]:
df.to_csv(
    "feature_engineered_transactions.csv",
    index=False
)

In [4]:
features = [
    "src_tx_count_past",
    "dst_tx_count_past",
    "src_to_dst_count_past",
    "dst_to_src_count_past",
    "time_since_src_last",
    "time_since_dst_last",
    "src_value_sum_past",
    "dst_value_sum_past",
    "log_transaction_value"
]

print(df[features].describe())

c:\Users\aryak\anaconda3\envs\nft-research\Lib\site-packages\pandas\core\nanops.py:1036: RuntimeWarning: overflow encountered in cast
  result = result.astype(dtype, copy=False)
c:\Users\aryak\anaconda3\envs\nft-research\Lib\site-packages\pandas\core\nanops.py:1036: RuntimeWarning: overflow encountered in cast
  result = result.astype(dtype, copy=False)


       src_tx_count_past  dst_tx_count_past  src_to_dst_count_past  \
count       2.713386e+06       2.713386e+06           2.713386e+06   
mean        3.854138e+03       9.914047e+01           2.664672e+00   
std         1.783007e+04       3.438955e+02           2.691849e+01   
min         0.000000e+00       0.000000e+00           0.000000e+00   
25%         2.100000e+01       4.000000e+00           0.000000e+00   
50%         7.900000e+01       1.900000e+01           0.000000e+00   
75%         2.850000e+02       7.700000e+01           0.000000e+00   
max         1.392410e+05       1.022500e+04           9.520000e+02   

       dst_to_src_count_past  time_since_src_last  time_since_dst_last  \
count           2.713386e+06         2.713386e+06         2.713386e+06   
mean            4.479790e-02         8.908772e+04         1.246666e+05   
std             2.101246e+00         4.352716e+05         5.634454e+05   
min             0.000000e+00         0.000000e+00         0.000000e+00   

In [5]:
fraud = df[
    df["is_wash_trading"] == 1
]

normal = df[
    df["is_wash_trading"] == 0
]

for col in features:

    print("\n" + "="*50)
    print(col)

    print(
        "Fraud median:",
        fraud[col].median()
    )

    print(
        "Normal median:",
        normal[col].median()
    )


src_tx_count_past
Fraud median: 186.0
Normal median: 79.0

dst_tx_count_past
Fraud median: 119.0
Normal median: 19.0

src_to_dst_count_past
Fraud median: 12.0
Normal median: 0.0

dst_to_src_count_past
Fraud median: 0.0
Normal median: 0.0

time_since_src_last
Fraud median: 0.0
Normal median: 3294.0

time_since_dst_last
Fraud median: 0.0
Normal median: 1560.0

src_value_sum_past
Fraud median: 7.164706e+19
Normal median: 2.8528685e+19

dst_value_sum_past
Fraud median: 7.16e+19
Normal median: 6.181973e+18

log_transaction_value
Fraud median: 39.54941168900694
Normal median: 39.5359886686748


In [7]:
features = [
    'src_tx_count_past',
    'dst_tx_count_past',
    'src_to_dst_count_past',
    'time_since_src_last',
    'time_since_dst_last'
]

for col in features:
    print("\n", col)
    print(df[col].quantile([0.5,0.75,0.9,0.95,0.99]))


 src_tx_count_past
0.50        79.00
0.75       285.00
0.90       995.00
0.95     11156.75
0.99    112107.15
Name: src_tx_count_past, dtype: float64

 dst_tx_count_past
0.50      19.00
0.75      77.00
0.90     227.00
0.95     416.00
0.99    1126.15
Name: dst_tx_count_past, dtype: float64

 src_to_dst_count_past
0.50     0.0
0.75     0.0
0.90     1.0
0.95     5.0
0.99    49.0
Name: src_to_dst_count_past, dtype: float64

 time_since_src_last
0.50       3231.0
0.75      32469.0
0.90     155687.0
0.95     366285.0
0.99    1559756.0
Name: time_since_src_last, dtype: float64

 time_since_dst_last
0.50       1513.0
0.75      46010.0
0.90     237738.5
0.95     546750.0
0.99    2123820.4
Name: time_since_dst_last, dtype: float64
